# Régression linéaire et résultats

En utilisant le notebook « Nettoyage des données », nous allons effectuer une régression linéaire sur les données nettoyées.

In [45]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


In [ ]:
import s3fs
import pandas as pd
import tempfile
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

MY_BUCKET = "gbern13"

FILE_PATH_S3 = f"{MY_BUCKET}/diffusion/PISA2022/student_questionnaire/CY08MSP_STU_QQQ.SAS7BDAT"
with fs.open(FILE_PATH_S3, "rb") as s3f:
    df = pd.read_sas(s3f)

print(df.head())

In [47]:
df.shape

(613744, 1278)

In [48]:
variables=['CNTSTUID','ST004D01T','ESCS','WORKPAY','WORKHOME','ICTRES','BULLIED',
'SCHRISK','FAMSUP','TEACHSUP','SCHSUST','CREATOOS','CREATAS','EXPO21ST','FEELSAFE','DISCLIM','COGACMCO']
## Réduction du dataset à ces variables 
data=df[variables].copy()
## Nouveaux noms des variables 
renom=['CNTSTUID','gender','society','work','household_care','digital_home',
'social','school_safety','family_support','teacher_support','school_support','creative_out',
'creative_at','math_reasoning','safety','climate_math','math_thinking']
## Renommage des variables
data.columns=renom
data.head()

,CNTSTUID,gender,society,work,household_care,digital_home,social,school_safety,family_support,teacher_support,school_support,creative_out,creative_at,math_reasoning,safety,climate_math,math_thinking
0,800001.0,1.0,1.1112,0.0,10.0,4.9507,-1.2280,-0.6386,1.8355,1.5558,NaN,NaN,4.1226,2.4031,1.1246,0.6387,2.4962
1,800002.0,2.0,-3.0507,NaN,NaN,-3.4930,1.3336,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.1246,NaN,NaN
2,800003.0,2.0,-0.1867,0.0,0.0,0.4307,NaN,-0.6386,NaN,NaN,NaN,NaN,NaN,NaN,0.8637,-0.8615,NaN
3,800005.0,1.0,-3.2198,0.0,10.0,-2.1392,0.9885,-0.6386,-0.7468,1.5558,1.5382,1.4468,1.0191,0.3556,-0.7560,0.4426,-0.1216
4,800006.0,1.0,-1.0548,0.0,4.0,-0.5542,-1.2280,-0.6386,-0.5122,0.1475,0.2241,1.8557,1.6583,-1.0257,1.1246,0.4029,0.7927


In [ ]:
FILE_PATH_S3 = f"{MY_BUCKET}/diffusion/PISA2022/student_questionnaire/CY08MSP_STU_QQQ.SAS7BDAT"
with fs.open(FILE_PATH_S3, "rb") as s3f:
    creat_df = pd.read_sas(s3f)
creat_df = creat_df[['CNTSTUID','PV1CRTH_NC']]
creat_df.head()

,CNTSTUID,PV1CRTH_NC
0,800001.0,2.9
1,800002.0,4.6
2,800003.0,6.5
3,800005.0,0.4
4,800006.0,14.3


La matrice de corrélation dans le notebook précédent montre que les variables « creative_out » et « creative_at » sont fortement corrélées (corrélation de 0,75), de même que « digital_home » et « society ». Nous proposons de remplacer les deux premières par leur moyenne (puisque l'on s'attendait à ce qu'elles soient similaires) et, pour les deux dernières, de vérifier le VIF.

In [50]:
data['creative_total'] = (data['creative_at']+data['creative_out'])/2
data.drop(['creative_at','creative_out'],axis=1,inplace=True)

In [60]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = data.drop('CNTSTUID', axis=1)

X_clean = X.dropna()

vif = pd.DataFrame()
vif['Variable'] = X_clean.columns
vif['VIF'] = [variance_inflation_factor(X_clean.values, i) for i in range(X_clean.shape[1])]

vif = vif.sort_values('VIF', ascending=False)
print(vif.to_string(index=False))

       Variable      VIF
 household_care 2.960955
         gender 2.847606
   digital_home 2.315628
        society 2.267123
           work 1.473303
  math_thinking 1.345951
         social 1.257598
teacher_support 1.250780
 creative_total 1.246411
  school_safety 1.235392
 math_reasoning 1.207040
   climate_math 1.169415
 family_support 1.162876
 school_support 1.149549
         safety 1.106986


On remarque que les deux variables « society » et « digital_home » ont un VIF faible. Même si les variables sont corrélées, elles apportent des informations différentes à la régression. Nous proposons donc de conserver les deux variables dans le modèle.

In [61]:
final_df = pd.merge(data, creat_df, on='CNTSTUID', how='inner')
final_df.shape

(499843, 17)

In [62]:
final_df.head()

,CNTSTUID,gender,society,work,household_care,digital_home,social,school_safety,family_support,teacher_support,school_support,math_reasoning,safety,climate_math,math_thinking,creative_total,PV1CRTH_NC
0,800001.0,1.0,1.1112,0.0,10.0,4.9507,-1.2280,-0.6386,1.8355,1.5558,NaN,2.4031,1.1246,0.6387,2.4962,NaN,2.9
1,800002.0,2.0,-3.0507,NaN,NaN,-3.4930,1.3336,NaN,NaN,NaN,NaN,NaN,1.1246,NaN,NaN,NaN,4.6
2,800003.0,2.0,-0.1867,0.0,0.0,0.4307,NaN,-0.6386,NaN,NaN,NaN,NaN,0.8637,-0.8615,NaN,NaN,6.5
3,800005.0,1.0,-3.2198,0.0,10.0,-2.1392,0.9885,-0.6386,-0.7468,1.5558,1.5382,0.3556,-0.7560,0.4426,-0.1216,1.23295,0.4
4,800006.0,1.0,-1.0548,0.0,4.0,-0.5542,-1.2280,-0.6386,-0.5122,0.1475,0.2241,-1.0257,1.1246,0.4029,0.7927,1.75700,14.3


Alors, on peut faire une première régression linéaire.

In [63]:
X = final_df.drop(['PV1CRTH_NC', 'CNTSTUID'], axis=1)
y = final_df['PV1CRTH_NC'] 

df_clean = pd.concat([X, y], axis=1).dropna()
X_clean = df_clean.drop('PV1CRTH_NC', axis=1)
y_clean = df_clean['PV1CRTH_NC']

X_const = sm.add_constant(X_clean)

# Ajustar o modelo
model = sm.OLS(y_clean, X_const).fit()

# Ver resultados
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             PV1CRTH_NC   R-squared:                       0.286
Model:                            OLS   Adj. R-squared:                  0.286
Method:                 Least Squares   F-statistic:                     2701.
Date:                Sun, 28 Dec 2025   Prob (F-statistic):               0.00
Time:                        00:11:23   Log-Likelihood:            -3.8684e+05
No. Observations:              101038   AIC:                         7.737e+05
Df Residuals:                  101022   BIC:                         7.739e+05
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              32.5155      0.131    2

La variable « math_reasoning » n'est pas significative. On la supprime.

In [64]:
X = final_df.drop(['PV1CRTH_NC', 'CNTSTUID'], axis=1) 
y = final_df['PV1CRTH_NC'] 

df_clean = pd.concat([X, y], axis=1).dropna()
X_clean = df_clean.drop(['PV1CRTH_NC','math_reasoning'], axis=1)
y_clean = df_clean['PV1CRTH_NC']

X_const = sm.add_constant(X_clean)


model = sm.OLS(y_clean, X_const).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             PV1CRTH_NC   R-squared:                       0.286
Model:                            OLS   Adj. R-squared:                  0.286
Method:                 Least Squares   F-statistic:                     2894.
Date:                Sun, 28 Dec 2025   Prob (F-statistic):               0.00
Time:                        00:11:23   Log-Likelihood:            -3.8684e+05
No. Observations:              101038   AIC:                         7.737e+05
Df Residuals:                  101023   BIC:                         7.739e+05
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              32.5077      0.131    2

Quelques commentaires sur la régression :
Le fait d'être une femme diminue le score de créativité — cela peut être dû à un biais lié à une variable omise.
Les variables `society`, `digital_home`, `family_support`, `household_care`, `school_support` et `safety` ont des coefficients positifs, ce qui est logique.
La variable `work` a un coefficient négatif ; cela peut s'expliquer par le fait que, si l'on a plus de temps pour soi, on a plus de temps pour créer.
`math_thinking` a un coefficient négatif, ce qui n'est pas clair. Peut‑être qu'un raisonnement logique diminue la créativité.

Maintenant, pour un effet quadratique, on ajoute la variable digital_home^2

In [65]:
final_df['digital_home2'] = final_df['digital_home']**2

In [66]:
X = final_df.drop(['PV1CRTH_NC', 'CNTSTUID'], axis=1) 
y = final_df['PV1CRTH_NC']

df_clean = pd.concat([X, y], axis=1).dropna()
X_clean = df_clean.drop(['PV1CRTH_NC','math_reasoning'], axis=1)
y_clean = df_clean['PV1CRTH_NC']

X_const = sm.add_constant(X_clean)

model = sm.OLS(y_clean, X_const).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             PV1CRTH_NC   R-squared:                       0.292
Model:                            OLS   Adj. R-squared:                  0.292
Method:                 Least Squares   F-statistic:                     2783.
Date:                Sun, 28 Dec 2025   Prob (F-statistic):               0.00
Time:                        00:11:24   Log-Likelihood:            -3.8641e+05
No. Observations:              101038   AIC:                         7.729e+05
Df Residuals:                  101022   BIC:                         7.730e+05
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              33.0833      0.131    2

Ce tableau nous montre un gain décroissant de créativité à partir d'ajout d'accés à information. En effet, la valeur optimal serait 0.7841/(2*0.3784)=1.03.